
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo — Data Imputation and Transformation Pipeline

In this demo, you will work through a complete data preparation workflow for machine learning.

Starting from a raw, noisy dataset, you will progressively clean and transform the data—fixing data types, handling invalid values, addressing missing data, encoding categorical features, and preparing train and test datasets with proper scaling and imputation.

**Learning Objectives**

By the end of this demo, you will be able to:

- Coerce columns to the correct data types based on feature and target requirements  
- Identify and remove invalid or erroneous values in numeric columns  
- Analyze missing data and drop columns or rows based on defined thresholds  
- Impute boolean and string missing values using appropriate default strategies  
- Understand how Spark ML handles categorical missing values using `StringIndexer`  
- Encode categorical features using a `StringIndexer → OneHotEncoder → VectorAssembler` pipeline  
- Apply ordered indexing for ordinal categorical features  
- Split the dataset into training and test sets for modeling  
- Standardize numeric features using a scaler fit on the training set only  
- Impute numeric missing values using a strategy fit on the training set to avoid data leakage

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #F44336; background: #FFEBEE; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #C62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong> — <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976D2; text-decoration: underline;">How to select an environment version</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

### Classroom Setup
Run the following cell to configure your working environment for this course.

This setup will:
- Initialize the `DA` object (Databricks Academy helper)
- Configure your **default catalog** and **schema**
- Provision any supporting configuration needed for this demo

**NOTE:** The `DA` object is only available in Databricks Academy courses.

In [0]:
%run ../Includes/Classroom-Setup-2.1

**Other Conventions:**

Throughout this demo, we will refer to the object `DA`. This object, provided by Databricks Academy, contains variables such as your username, catalog name, schema name, working directory, and dataset locations. Run the code block below to view these details:

In [0]:
print(f"Username:          {DA.username}")
print(f"Catalog Name:      {DA.catalog_name}")
print(f"Schema Name:       {DA.schema_name}")
print(f"Working Directory: {DA.paths.working_dir}")
print(f"Dataset Location:  {DA.paths.datasets}")


## Data Cleaning and Imputation

We start by loading the raw dataset and walking through the full data preparation workflow: fixing data types, handling outliers, and addressing missing values. This prepares a clean dataset ready for feature engineering and modeling.

In [0]:
# Load dataset with spark
shared_volume_name = 'telco' # From Marketplace
csv_name = 'telco-customer-churn-noisy' # CSV file name
dataset_path = f"{DA.paths.datasets.telco}/{shared_volume_name}/{csv_name}.csv" # Full path

telco_df = spark.read.csv(dataset_path, header="true", inferSchema="true", multiLine="true", escape='"')

display(telco_df)


### Coerce / Fix Data Types

Even when Spark infers a schema, some columns may land in the wrong type. Correcting data types improves memory efficiency and ensures compatibility with Spark ML estimators.

We will:

* Convert **`SeniorCitizen`** and **`Churn`** binary columns to `BooleanType`.
* Convert **`tenure`** to `LongType` using `.selectExpr`.
* Convert **`Partner`**, **`Dependents`**, **`PhoneService`**, and **`PaperlessBilling`** to `BooleanType` using `spark.sql`.

In [0]:
from pyspark.sql.types import BooleanType, ShortType, IntegerType
from pyspark.sql.functions import col, when


binary_columns = ["SeniorCitizen", "Churn"]
telco_customer_churn_df = telco_df
for column in binary_columns:
    telco_customer_churn_df = telco_customer_churn_df.withColumn(column, col(column).cast(BooleanType()))

telco_customer_churn_df.select(*binary_columns).printSchema()

In [0]:
# PhoneService, PaperlessBilling, Partner, Dependents to boolean using spark.sql
telco_customer_churn_df.createOrReplaceTempView("telco_customer_churn_temp_view")

telco_customer_casted_df = spark.sql("""
    SELECT
        customerID,
        BOOLEAN(Dependents),
        BOOLEAN(Partner),
        BOOLEAN(PhoneService),
        BOOLEAN(PaperlessBilling),
        *
        EXCEPT (customerID, Dependents, Partner, PhoneService, PaperlessBilling, Churn),
        Churn
    FROM telco_customer_churn_temp_view
""")

telco_customer_casted_df.select("Dependents", "Partner", "PaperlessBilling", "PhoneService").printSchema()

In [0]:
# Tenure to Long/Integer using .selectExpr
telco_customer_casted_df = telco_customer_casted_df.selectExpr("* except(tenure)", "cast(tenure as long) tenure")
telco_customer_casted_df.select("tenure").printSchema()


### Handle Invalid Values and Review Distributions

Outliers can distort model training and statistical summaries. We address them in two steps:

1. **Remove rows with negative `TotalCharges`** — these are clearly erroneous values since charges cannot be negative.
2. **Review the `PaymentMethod` distribution** — understanding how customers are spread across payment types helps inform encoding decisions later.

**Filtering out negative TotalCharges**

Negative values in `TotalCharges` are data quality issues. We remove these rows while preserving rows where `TotalCharges` is null (which will be handled separately in the imputation step).

In [0]:
from pyspark.sql.functions import col


# Remove customers with negative TotalCharges
TotalCharges_cutoff = 0

telco_no_outliers_df = telco_customer_casted_df.filter(
    (col("TotalCharges") > TotalCharges_cutoff) |
    (col("TotalCharges").isNull())  # Keep Nulls — handled in imputation step
)

**Exploring Payment Method Distribution**

Let's review how customers are distributed across `PaymentMethod` categories. This is a useful step before encoding — it helps identify whether any categories are rare enough to require special handling.

**Note:** In this dataset, all `PaymentMethod` categories are well-represented. We do not remove any rows based on frequency here. In a real-world scenario with a high-cardinality feature containing very rare categories, you might consider grouping them under a common label such as `"Other"` before encoding.

In [0]:
from pyspark.sql.functions import col, count, avg


group_var = "PaymentMethod"
stats_df = telco_no_outliers_df.groupBy(group_var) \
                      .agg(count("*").alias("Total"),
                           avg("MonthlyCharges").alias("MonthlyCharges")) \
                      .orderBy(col("Total").desc())

display(stats_df)

Save the cleaned dataset as the silver table before moving to missing value handling.

In [0]:
telco_customer_name_full = "telco_customer_full"
telco_customer_full_silver = f"{telco_customer_name_full}_silver"

telco_no_outliers_df.write.mode("overwrite").option("mergeSchema", True).saveAsTable(telco_customer_full_silver)


### Handling Missing Values

Missing data is one of the most common challenges in real-world ML datasets. Our approach:

1. **Identify** which columns have missing values and how many.
2. **Drop columns** with more than 60% missing data — they carry too little signal to be useful.
3. **Drop rows** where more than 80% of fields are null — these rows add little to model training.
4. **Impute** remaining missing values with appropriate defaults.

In [0]:
from pyspark.sql.functions import (
    col,
    when,
    count,
    concat_ws,
    collect_list,
    sum,
    length,
    trim,
    lower,
)


def calculate_missing(input_df, show=True):
    """
    Helper function to calculate and display missing data per column.
    """
    missing_df_ = input_df.agg(
        *[
            sum(
                when(
                    col(c).isNull()
                    | (length(trim(col(c).cast("string"))) == 0)
                    | lower(trim(col(c).cast("string"))).isin(["none", "null"]),
                    1,
                ).otherwise(0)
            ).alias(c)
            for c in input_df.columns
        ]
    )

    def TransposeDF(df, columns, pivotCol):
        """Helper function to transpose spark dataframe"""
        columnsValue = list(
            map(lambda x: str("'") + str(x) + str("',") + str(x), columns)
        )
        stackCols = ",".join(x for x in columnsValue)
        df_1 = df.selectExpr(
            pivotCol, "stack(" + str(len(columns)) + "," + stackCols + ")"
        ).select(pivotCol, "col0", "col1")
        final_df = (
            df_1.groupBy(col("col0"))
            .pivot(pivotCol)
            .agg(concat_ws("", collect_list(col("col1"))))
            .withColumnRenamed("col0", pivotCol)
        )
        return final_df

    missing_df_out_T = TransposeDF(
        spark.createDataFrame([{"Column": "Number of Missing Values"}]).join(
            missing_df_
        ),
        missing_df_.columns,
        "Column",
    ).withColumn(
        "Number of Missing Values", col("Number of Missing Values").cast("long")
    )

    if show:
        display(missing_df_out_T.orderBy("Number of Missing Values", ascending=False))

    return missing_df_out_T


missing_df = calculate_missing(telco_no_outliers_df)

**Drop columns with more than 60% missing data**

Columns missing more than 60% of their values provide too little signal and introduce noise. We identify and drop them.

In [0]:
per_thresh = 0.6  # Drop if column has more than 60% missing data

N = telco_no_outliers_df.count()
to_drop_missing = [x.asDict()['Column'] for x in missing_df.select("Column").where(col("Number of Missing Values") / N >= per_thresh).collect()]

print(f"Dropping columns {to_drop_missing} for more than {per_thresh * 100}% missing data")
telco_no_missing_df = telco_no_outliers_df.drop(*to_drop_missing)
display(telco_no_missing_df)

**Drop rows where most fields are missing**

Rows with more than 80% of their fields missing provide very little information. We use `.na.drop()` with a threshold to remove them.

In [0]:
n_cols = len(telco_no_missing_df.columns)
telco_no_missing_df = telco_no_missing_df.na.drop(how='any', thresh=round(n_cols * .80))

print(f"Count — Before row drop: {telco_no_outliers_df.count()} / After: {telco_no_missing_df.count()}")


#### Impute Missing Data

After dropping high-missing columns and sparse rows, we fill in the remaining missing values with sensible defaults.

**Replace boolean missing values with `False`**

In [0]:
from pyspark.sql.types import BooleanType


bool_cols = [c.name for c in telco_no_missing_df.schema.fields if (c.dataType == BooleanType())]
telco_imputed_df = telco_no_missing_df.na.fill(value=False, subset=bool_cols)

**Replace string missing values with `No`**

For service-related string columns (e.g., `OnlineSecurity`, `TechSupport`), a missing value typically means the service is not subscribed to. We impute these with `"No"`. Columns with meaningful categories (`gender`, `Contract`, `PaymentMethod`) are excluded and handled separately.

In [0]:
from pyspark.sql.types import StringType


to_exclude = ["customerID", "gender", "Contract", "PaymentMethod"]
string_cols = [c.name for c in telco_no_missing_df.drop(*to_exclude).schema.fields if c.dataType == StringType()]

telco_imputed_df = telco_imputed_df.na.fill(value='No', subset=string_cols)

**Handling missing values in categorical columns**

For categorical columns such as `gender`, `Contract`, and `PaymentMethod`, we do not impute with a fixed string value because these columns have meaningful, distinct categories.

Instead, Spark ML's `StringIndexer` handles nulls automatically when `handleInvalid` is set to `"keep"`. This treats null as its own separate category during indexing — an approach that works well for tree-based models, which can effectively learn from the presence of missing data as a signal.

**When is mode imputation an alternative?**
Mode imputation (replacing nulls with the most frequent category) is appropriate when:
- The model cannot handle null-derived categories.
- Missing data is likely Missing At Random (MAR) and the mode is a reasonable proxy.

In this demo, we rely on `StringIndexer`'s built-in null handling during the encoding step.

In [0]:
# Confirm remaining missing values after imputation
calculate_missing(telco_imputed_df)

In [0]:
telco_imputed_df.write.mode("overwrite").saveAsTable(f"{DA.catalog_name}.{DA.schema_name}.telco_imputed_silver")


## Encoding Categorical Features

Most machine learning algorithms cannot accept raw string features. We need to convert categorical columns into numeric representations. Here we use the standard Spark MLlib workflow:

**`StringIndexer → OneHotEncoder → VectorAssembler`**

* **`StringIndexer`** — converts each unique string value to a numeric index.
* **`OneHotEncoder`** — converts the indexed column into a binary sparse vector, one position per category.
* **`VectorAssembler`** — combines one or more feature vectors into a single feature vector ready for a model.

Setting `handleInvalid="keep"` in `StringIndexer` ensures that null values are treated as a separate category rather than causing an error.

In [0]:
sample_df = telco_imputed_df.select("Contract").distinct()
sample_df.show()

In [0]:
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import col


# StringIndexer
string_cols = ["Contract"]
index_cols = [column + "_index" for column in string_cols]

string_indexer = StringIndexer(inputCols=string_cols, outputCols=index_cols, handleInvalid="keep")
string_indexer_model = string_indexer.fit(sample_df)
indexed_df = string_indexer_model.transform(sample_df)

indexed_df.show()

Once indexed, we apply `OneHotEncoder` to produce binary vector representations.

In [0]:
from pyspark.ml.feature import OneHotEncoder


ohe_cols = [column + "_ohe" for column in string_cols]

ohe = OneHotEncoder(inputCols=index_cols, outputCols=ohe_cols, handleInvalid="keep")
ohe_model = ohe.fit(indexed_df)
ohe_df = ohe_model.transform(indexed_df)
ohe_df.show()

In [0]:
from pyspark.ml.feature import VectorAssembler


selected_ohe_cols = ["Contract_ohe"]

assembler = VectorAssembler(inputCols=selected_ohe_cols, outputCol="features")
result_df_dense = assembler.transform(ohe_df)

result_df_dense.select("Contract", "features").show(truncate=False)


### Ordered Indexing

Some categorical columns are **ordinal** — they have a natural order that should be preserved. Standard one-hot encoding ignores this order. For example, `Contract` has a meaningful progression: `Month-to-month → One year → Two year`.

For ordinal features used in tree-based models, manually mapping categories to ordered integer values can improve model interpretability and sometimes performance.

In [0]:
ordinal_cat = "Contract"
telco_imputed_df.select(ordinal_cat).distinct().show(truncate=False)

In [0]:
# Define the ordered category-to-index mapping
ordered_list = [
    "Month-to-month",
    "One year",
    "Two year"
]

ordinal_dict = {category: f"{index + 1}" for index, category in enumerate(ordered_list)}
print(ordinal_dict)

In [0]:
from pyspark.sql.functions import expr


ordinal_df = (
    telco_imputed_df
    .withColumn(f"{ordinal_cat}_ord", col(ordinal_cat))
    .replace(to_replace=ordinal_dict, subset=[f"{ordinal_cat}_ord"])
    .withColumn(f"{ordinal_cat}_ord", col(f"{ordinal_cat}_ord").cast('int'))
)

display(ordinal_df.select(ordinal_cat, f"{ordinal_cat}_ord"))


## Splitting Data

Before applying any transformations that learn from the data (such as scaling or imputation), we split the dataset into training and test sets. This prevents **data leakage** — the test set must remain unseen during any fitting step.

We use a reproducible 80/20 split with a fixed seed. Writing both sets as Delta tables ensures the split is stable across notebook runs.

In [0]:
# Split with 80% in train and 20% in test
train_df, test_df = telco_imputed_df.randomSplit([.8, .2], seed=42)

In [0]:
# Materialize both sets as Delta tables
train_df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{DA.catalog_name}.{DA.schema_name}.telco_customers_train")
test_df.write.mode("overwrite").option("overwriteSchema", True).saveAsTable(f"{DA.catalog_name}.{DA.schema_name}.telco_customers_baseline")


### Standardize Features in a Training Set

Feature scaling brings numeric features onto a comparable scale, which is important for distance-based and gradient-based models. We use `RobustScaler`, which is less sensitive to outliers than `StandardScaler` because it scales based on the interquartile range (IQR) rather than mean and standard deviation.

**Key principle:** The scaler is **fit on the training set only**. The same fitted scaler is then applied to both training and test sets. This prevents test set statistics from influencing the transformation.

In [0]:
from pyspark.ml.feature import RobustScaler, VectorAssembler


num_cols_to_scale = ["MonthlyCharges"]
assembler = VectorAssembler().setInputCols(num_cols_to_scale).setOutputCol("numerical_assembled")

train_assembled_df = assembler.transform(train_df.select(*num_cols_to_scale))
test_assembled_df = assembler.transform(test_df.select(*num_cols_to_scale))

# Fit the scaler on the training set only
scaler = RobustScaler(inputCol="numerical_assembled", outputCol="numerical_scaled")
scaler_fitted = scaler.fit(train_assembled_df)

# Apply to both training and test sets
train_scaled_df = scaler_fitted.transform(train_assembled_df)
test_scaled_df = scaler_fitted.transform(test_assembled_df)

In [0]:
print("Training set (scaled):")
train_scaled_df.show(5)

In [0]:
print("Test set (scaled):")
test_scaled_df.show(5)


### Numeric Missing Value Imputation

Numeric features may still contain missing values after the earlier cleaning steps. Before passing data to a model, these must be filled with a representative value.

**Mean vs. Median — when to use which:**

| Strategy | When to Use |
|----------|-------------|
| **Mean** | Feature is approximately normally distributed with no significant outliers |
| **Median** | Feature is skewed or contains outliers — median is more robust |

For `tenure`, which can be skewed toward newer or longer-tenured customers, we use the **median** strategy.

As with scaling, the imputer must be **fit on the training set only** and then applied to both sets to avoid data leakage.

In [0]:
from pyspark.ml.feature import Imputer


# Define the imputer for tenure using median strategy
tenure_imputer = Imputer(
    inputCols=["tenure"],
    outputCols=["tenure_imputed"],
    strategy="median"
)

# Fit on training set only
tenure_imputer_fitted = tenure_imputer.fit(train_df.select("tenure"))

# Apply to both training and test sets
train_tenure_imputed_df = tenure_imputer_fitted.transform(train_df.select("tenure"))
test_tenure_imputed_df = tenure_imputer_fitted.transform(test_df.select("tenure"))

In [0]:
print("Training set — tenure imputed:")
train_tenure_imputed_df.show(5)

In [0]:
print("Test set — tenure imputed:")
test_tenure_imputed_df.show(5)

## Conclusion

In this demo, you implemented an end-to-end data preparation workflow for machine learning using Spark.

You were able to:

- Correct data types to ensure compatibility with Spark ML  
- Identify and handle invalid values in numeric features  
- Analyze and address missing data using appropriate strategies  
- Encode categorical features using indexing and one-hot encoding  
- Split the dataset into training and test sets  
- Apply scaling and numeric imputation using transformations fit on the training data only  

These steps form the foundation of a reliable and reproducible data preparation process. In the next demo, you will build on this work by assembling these transformations into a structured machine learning pipeline.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>